In [181]:
import pandas as pd

In [182]:
books_df = pd.read_csv('../data/challenge_427_book_sales.csv')

books_df.head()

,Date,BookTitle
0,2024-01-01,Lord of the Flies
1,2024-01-01,1984
2,2024-01-01,Lord of the Flies
3,2024-01-01,1984
4,2024-01-01,The Great Gatsby


In [183]:
time_df = pd.read_csv('../data/challenge_427_time_periods.csv')

time_df.head()

,Start_YMD,End_YMD,MonthToReport
0,2023-12-01,2024-05-31,1
1,2023-11-01,2024-04-30,2
2,2023-10-01,2024-03-31,3


In [184]:
# convert strings to datetimes

books_df['Date'] = pd.to_datetime(books_df['Date'])
time_df['Start_YMD'] = pd.to_datetime(time_df['Start_YMD'])
time_df['End_YMD'] = pd.to_datetime(time_df['End_YMD'])

In [185]:
# cross join so each time period is on the same row as each sale for comparison, filter to books relevant to each period

df = pd.merge(time_df, books_df, how='cross')
df = df[(df['Date'] >= df['Start_YMD']) & (df['Date'] <= df['End_YMD'])]

df.info()

<class 'pandas.DataFrame'>
Index: 91014 entries, 0 to 181767
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Start_YMD      91014 non-null  datetime64[us]
 1   End_YMD        91014 non-null  datetime64[us]
 2   MonthToReport  91014 non-null  int64         
 3   Date           91014 non-null  datetime64[us]
 4   BookTitle      91014 non-null  str           
dtypes: datetime64[us](3), int64(1), str(1)
memory usage: 4.2 MB


In [186]:
# count total books and sales per book in each period

all_books_in_period_df = df.groupby(['Start_YMD', 'End_YMD'])['BookTitle'].count().reset_index(name='AllBookSalesInDateRange')
top_book_in_period_df = df.groupby(['Start_YMD', 'End_YMD', 'BookTitle'])['BookTitle'].count().reset_index(name='BookTitleSalesInDateRange')

In [187]:
# filter df to only the top book per period

top_book_in_period_df = top_book_in_period_df.sort_values(by='BookTitleSalesInDateRange', ascending=False)
idx = top_book_in_period_df.groupby('Start_YMD')['BookTitleSalesInDateRange'].idxmax()

top_book_in_period_df = top_book_in_period_df.loc[idx]
top_book_in_period_df

,Start_YMD,End_YMD,BookTitle,BookTitleSalesInDateRange
4,2023-10-01,2024-03-31,The Corrections,3689
13,2023-11-01,2024-04-30,The Great Gatsby,3873
18,2023-12-01,2024-05-31,Lord of the Flies,3993


In [188]:
final_df = pd.merge(all_books_in_period_df, top_book_in_period_df, how='inner', on=['Start_YMD', 'End_YMD'])
final_df

,Start_YMD,End_YMD,AllBookSalesInDateRange,BookTitle,BookTitleSalesInDateRange
0,2023-10-01,2024-03-31,29175,The Corrections,3689
1,2023-11-01,2024-04-30,30441,The Great Gatsby,3873
2,2023-12-01,2024-05-31,31398,Lord of the Flies,3993
